# Dataset

In [1]:
from datasets import load_dataset

dataset = load_dataset("FinanceMTEB/financial_phrasebank")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label_text', 'label'],
        num_rows: 1264
    })
    test: Dataset({
        features: ['text', 'label_text', 'label'],
        num_rows: 1000
    })
})


- The dataset contains 2.264 rows which is the **number of instances with 100% annotator agreement** from the original dataset
- The dataset is split poorly (train: 1.264 rows (55%), test: 1.000 rows(45%)) and will be split again following the logic **train: 70%**, **validation: 10%**, **test: 20%**.

In [2]:
from datasets import concatenate_datasets

full_dataset = concatenate_datasets([dataset["train"], dataset["test"]])

print(full_dataset)

Dataset({
    features: ['text', 'label_text', 'label'],
    num_rows: 2264
})


In [3]:
len(full_dataset.unique("label"))

3

In [4]:
from datasets import ClassLabel

full_dataset = full_dataset.cast_column("label", ClassLabel(num_classes=3))

In [5]:
from datasets import DatasetDict

temp_split = full_dataset.train_test_split(
    test_size=0.2, 
    seed=42, 
    stratify_by_column="label"
)

final_split = temp_split['train'].train_test_split(
    test_size=0.125, 
    seed=42, 
    stratify_by_column="label"
)

dataset_dict = DatasetDict({
    'train': final_split['train'],
    'val': final_split['test'],
    'test': temp_split['test']
})

print(dataset_dict)

DatasetDict({
    train: Dataset({
        features: ['text', 'label_text', 'label'],
        num_rows: 1584
    })
    val: Dataset({
        features: ['text', 'label_text', 'label'],
        num_rows: 227
    })
    test: Dataset({
        features: ['text', 'label_text', 'label'],
        num_rows: 453
    })
})


# Tokenizer

In [2]:
from src.data_loader import load_final_data

dataset_dict = load_final_data("financial_phrasebank_stratified")

dataset_dict

DatasetDict({
    train: Dataset({
        features: ['text', 'label_text', 'label'],
        num_rows: 1629
    })
    val: Dataset({
        features: ['text', 'label_text', 'label'],
        num_rows: 182
    })
    test: Dataset({
        features: ['text', 'label_text', 'label'],
        num_rows: 453
    })
})

In [3]:
dataset_dict["train"][0]["text"]

'CapMan , with offices in Helsinki , Stockholm , Copenhagen and Oslo , manages Nordic buyout , mezzanine , technology , life science and real estate funds with approximately EUR2 .6 bn in total capital .'

In [12]:
from transformers import AutoTokenizer

checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

tokenized = tokenizer(str(dataset_dict["train"][0]["text"]))
tokenized

{'input_ids': [101, 6178, 2386, 1010, 2007, 4822, 1999, 12331, 1010, 8947, 1010, 9664, 1998, 9977, 1010, 9020, 13649, 4965, 5833, 1010, 2033, 20715, 19105, 1010, 2974, 1010, 2166, 2671, 1998, 2613, 3776, 5029, 2007, 3155, 7327, 2099, 2475, 1012, 1020, 24869, 1999, 2561, 3007, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [14]:
from transformers import AutoTokenizer

checkpoint = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

tokenized = tokenizer(str(dataset_dict["train"][0]["text"]))
tokenized

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

{'input_ids': [101, 6178, 2386, 1010, 2007, 4822, 1999, 12331, 1010, 8947, 1010, 9664, 1998, 9977, 1010, 9020, 13649, 4965, 5833, 1010, 2033, 20715, 19105, 1010, 2974, 1010, 2166, 2671, 1998, 2613, 3776, 5029, 2007, 3155, 7327, 2099, 2475, 1012, 1020, 24869, 1999, 2561, 3007, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [4]:
from transformers import AutoTokenizer

checkpoint = "microsoft/deberta-v3-small"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

tokenized = tokenizer(str(dataset_dict["train"][0]["text"]))
tokenized

{'input_ids': [10436, 7491, 366, 275, 4030, 267, 22540, 366, 16339, 366, 16809, 263, 22377, 366, 8393, 17478, 35286, 366, 49894, 366, 835, 366, 432, 1693, 263, 609, 1978, 2145, 275, 2407, 14329, 445, 323, 765, 2165, 673, 267, 1047, 1909, 323], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [19]:
from transformers import AutoTokenizer

checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

tokenized = tokenizer(str(dataset_dict["train"]["text"]))
tokenized

{'input_ids': [101, 5930, 1006, 1031, 1005, 6178, 2386, 1010, 2007, 4822, 1999, 12331, 1010, 8947, 1010, 9664, 1998, 9977, 1010, 9020, 13649, 4965, 5833, 1010, 2033, 20715, 19105, 1010, 2974, 1010, 2166, 2671, 1998, 2613, 3776, 5029, 2007, 3155, 7327, 2099, 2475, 1012, 1020, 24869, 1999, 2561, 3007, 1012, 1005, 1010, 1000, 12316, 2102, 3840, 17419, 1012, 2484, 1010, 2289, 2012, 2410, 1024, 2382, 5971, 2713, 4518, 3863, 2713, 12316, 2102, 1005, 1055, 3007, 6089, 2154, 1999, 2414, 1010, 17419, 1012, 2656, 1010, 2289, 2006, 9317, 1010, 2244, 2656, 1010, 2289, 1010, 12316, 2102, 2097, 2907, 1037, 3007, 6089, 2154, 2005, 9387, 1998, 18288, 1999, 2414, 1012, 1000, 1010, 1005, 1996, 3820, 12919, 2015, 2256, 2146, 1011, 2744, 5386, 2007, 22098, 22108, 6125, 1012, 1005, 1010, 1005, 13672, 2700, 5887, 14194, 3472, 10913, 1010, 1050, 1012, 1044, 1012, 1011, 5146, 1037, 1012, 13672, 1010, 5766, 1997, 2326, 4923, 2586, 1010, 2038, 2042, 2700, 3472, 1997, 1996, 3639, 4923, 2586, 2473, 2005, 1996, 22

In [5]:
from transformers import AutoTokenizer

checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_function(example):
    return tokenizer(example["text"], truncation=True)

dataset_dict = dataset_dict.map(tokenize_function, batched=True)
dataset_dict

Map:   0%|          | 0/1629 [00:00<?, ? examples/s]

Map:   0%|          | 0/182 [00:00<?, ? examples/s]

Map:   0%|          | 0/453 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label_text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1629
    })
    val: Dataset({
        features: ['text', 'label_text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 182
    })
    test: Dataset({
        features: ['text', 'label_text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 453
    })
})

In [6]:
dataset_dict["train"][0]

{'text': 'CapMan , with offices in Helsinki , Stockholm , Copenhagen and Oslo , manages Nordic buyout , mezzanine , technology , life science and real estate funds with approximately EUR2 .6 bn in total capital .',
 'label_text': 'neutral',
 'label': 1,
 'input_ids': [101,
  6178,
  2386,
  1010,
  2007,
  4822,
  1999,
  12331,
  1010,
  8947,
  1010,
  9664,
  1998,
  9977,
  1010,
  9020,
  13649,
  4965,
  5833,
  1010,
  2033,
  20715,
  19105,
  1010,
  2974,
  1010,
  2166,
  2671,
  1998,
  2613,
  3776,
  5029,
  2007,
  3155,
  7327,
  2099,
  2475,
  1012,
  1020,
  24869,
  1999,
  2561,
  3007,
  1012,
  102],
 'token_type_ids': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  